In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
import seaborn as sns

from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor


In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
# Task 1: Write your code here:
food_path = os.path.join(path, 'Q1_data.csv')
df_food = pd.read_csv(food_path)

print(f"Dataset shape: {df_food.shape}")

In [ ]:
# Task 2: Write your code here:
df_food.head(10)

In [ ]:
# Task 3: Write your code here:
df_food.info()

In [ ]:
# Task 4: Write your code here:
df_food.describe()

In [ ]:
# Task 5: Write your code here:
plt.figure(figsize=(10, 5))
plt.hist(df_food['Delivery_Time'].dropna(), bins=50, edgecolor='black')
plt.title('Delivery Time Distribution')
plt.xlabel('Time')
plt.ylabel('Frequency')
plt.show()

In [ ]:
# Task 1: Write your code here:
# df_food.drop(df_food['Order_ID'], axis=1)
df_food = df_food.drop(['Order_ID'], axis=1)
df_food.head(5)

In [ ]:
# Task 2: Write your code here:
def check_missing_values(df):
  missing_values = df.isnull().sum()
  print("Missing Values per Column:")
  print(missing_values[missing_values > 0])
  if missing_values.any():
    print("\nHandle Missing Values as needed.")
  else:
    print("\nNo Missing Values Found.")

check_missing_values(df_food)

In [ ]:
df_clean = df_food.copy()
df_clean = df_clean.dropna(subset=['Delivery_Time'])
df_clean.head(5)

In [ ]:
cols=['Weather','Traffic_Level','Time_of_Day','Courier_Experience_yrs']
for col in cols:
  df_clean[col] = df_clean[col].fillna(df_clean[col].mode()[0])

In [ ]:
check_missing_values(df_clean)

In [ ]:
# Task 3: Write your code here:
def check_duplicates(df):
  duplicates = df_food.duplicated().sum()
  print(f"Number of Duplicate Samples: {duplicates}")
  if duplicates > 0:
    print("Dropping Duplicates...")
    df.drop_duplicates(inplace=True)
    print("Duplicates Dropped.")
  else:
    print("No Duplicate Samples Found.")

check_duplicates(df_clean)

In [ ]:
# df_clean.drop_duplicates(inplace=True)

In [ ]:
categorical_cols = df_clean.select_dtypes(include=["object"]).columns

print("Categorical Columns:", list(categorical_cols))

In [ ]:
for col in categorical_cols:
    le = LabelEncoder()
    df_clean[col] = le.fit_transform(df_clean[col].astype(str))

df_food.head()

In [ ]:
# Task 5: Write your code here:

numerical_cols = df_clean.select_dtypes(include=["int64", "float64"]).columns.drop("Delivery_Time")  # DON'T SCALE THE TARGET

scaler = StandardScaler()
df_clean[numerical_cols] = scaler.fit_transform(df_clean[numerical_cols])
df_clean.head()

In [ ]:
# Task 6: Write your code here:
def check_target_imbalance(df, target_column):
  print("Target Distribution:")
  print(df[target_column].value_counts(normalize=True))
  sns.countplot(x=df[target_column])
  plt.title("Delivery Time Distribution")
  plt.show()

check_target_imbalance(df_clean, "Delivery_Time")

In [ ]:
df_clean.columns

In [ ]:
# Task 1: Write your code here:
# feature_cols = ['Distance_km', 'Weather', 'Traffic_Level', 'Time_of_Day','Vehicle_Type', 'Preparation_Time_min', 'Courier_Experience_yrs']
# X = df_clean
# y = df_clean['price']

X = df_clean.drop("Delivery_Time", axis=1)
y = df_clean["Delivery_Time"]

# Stratified split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")
print(f"Delivery_Time in train: {y_train.sum()}, in test: {y_test.sum()}")

In [ ]:
from sklearn.model_selection import StratifiedKFold

n_splits = 3 # K=3 Folds

skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

In [ ]:
# scaler = StandardScaler()
# X_train_scaled = scaler.fit_transform(X_train)
# X_test_scaled = scaler.transform(X_test)

# print(f"\nScaled ranges - Min: {X_train_scaled.min():.2f}, Max: {X_train_scaled.max():.2f}")
# pd.DataFrame(X_train_scaled, columns=X_train.columns).head(3)

In [ ]:
# Task 2,3,4,5: Write your code here:
model = RandomForestRegressor(n_estimators=100, max_depth=20, random_state=42, n_jobs=-1)
model.fit(X_train, y_train)
print("Model trained!")

In [ ]:
from sklearn.metrics import mean_absolute_error

y_pred = model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)

print(f"MAE:  ${mae:,.2f}")

In [ ]:
# lr_accuracy = []
# lr_precision = []
# lr_recall = []
# lr_f1 = []

In [ ]:
# from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
# for fold_idx, (train_index, test_index) in enumerate(skf.split(X, y)):

#     # Calculate evaluation metrics
#     accuracy = accuracy_score(y_test, y_pred)
#     precision = precision_score(y_test, y_pred, zero_division=0)
#     recall = recall_score(y_test, y_pred, zero_division=0)
#     f1 = f1_score(y_test, y_pred, zero_division=0)

#     # Store results
#     lr_accuracy.append(accuracy)
#     lr_precision.append(precision)
#     lr_recall.append(recall)
#     lr_f1.append(f1)

In [ ]:
# print("LOGISTIC REGRESSION Performance:")
# print(f"  Accuracy:  {np.mean(lr_accuracy):.4f}")
# print(f"  Precision: {np.mean(lr_precision):.4f}")
# print(f"  Recall:    {np.mean(lr_recall):.4f}")
# print(f"  F1-Score:  {np.mean(lr_f1):.4f}")

In [ ]:
# Task 1: Write your code here:
# Feature importance
feature_cols = ['Distance_km', 'Weather', 'Traffic_Level', 'Time_of_Day','Vehicle_Type', 'Preparation_Time_min', 'Courier_Experience_yrs']

feature_importance = pd.DataFrame({
    'feature': feature_cols,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

plt.figure(figsize=(10, 6))
plt.barh(feature_importance['feature'], feature_importance['importance'])
plt.xlabel('Importance')
plt.title('Feature Importance')
plt.gca().invert_yaxis()
plt.show()

In [ ]:
# Task 2: Write your code here:
plt.figure(figsize=(6, 4))
plt.scatter(y_test, y_pred, alpha=0.6)
plt.plot(
    [y_test.min(), y_test.max()],
    [y_test.min(), y_test.max()],
    "r--",
    linewidth=2
)

In [ ]:
# Task Bonus: Write your code here: